<div style="text-align:right">
<img src="https://raw.githubusercontent.com/cayetanoguerra/redes-neuronales/main/notebooks/imgs/logo-curso.jpg" width="30%" style="max-width:100%; padding-right:20px;">
</div>

**Fundamentos de la Inteligencia Artificial y Programación de APIs para LLMs**

# **10. Razonamiento: de la cadena de pensamiento a los modelos que razonan**

<div class="colab-badge">
<a href="https://colab.research.google.com/github/cayetanoguerra/redes-neuronales/blob/main/notebooks/10-razonamiento.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>
<br><em>Para trabajar con tu propia copia, ábrelo en Colab y elige <b>Archivo → Guardar una copia en Drive</b>.</em>
</div>

En los notebooks anteriores vimos que un modelo de lenguaje genera su respuesta token a token, que podemos tener uno funcionando en nuestro propio ordenador con LM Studio y que nuestros programas pueden usarlo a través de su API. También vimos que se equivoca: a veces produce, con total seguridad, una respuesta que suena bien y es falsa.

En este notebook empezamos un bloque nuevo, dedicado a cómo se ha ido construyendo la «inteligencia» de estos sistemas por encima de la simple predicción de tokens. El primer paso es el **razonamiento**. Veremos que un mismo modelo, sin cambiar ni uno solo de sus parámetros, falla un problema cuando le pedimos el resultado y lo acierta cuando le pedimos que lo resuelva paso a paso. Entenderemos por qué ocurre y cómo esa observación, que empezó siendo un truco al escribir la petición, ha terminado dando lugar a los **modelos razonadores** actuales.

Esta vez no nos limitaremos a leer: haremos los experimentos con Qwen3, el modelo de 4B parámetros que instalamos en LM Studio. Todas las salidas que verás son respuestas reales de ese modelo.

## **1. ¿Puede «pensar» un predictor de tokens?**

Recordemos cómo responde un modelo de lenguaje. Recibe un contexto, calcula qué token podría venir a continuación, elige uno, lo añade al contexto y repite. No dispone de un borrador aparte ni de una memoria en la que guardar resultados parciales: **lo único que conserva de un paso al siguiente es el texto que ya está escrito**.

Ahora piensa en cómo resolvemos las personas dos tipos de preguntas:

- «¿Cuánto es 7 × 8?». Respondemos al instante, de memoria.
- «¿Cuánto es 347 × 286?». Nadie responde al instante. Cogemos papel y lápiz, hacemos multiplicaciones parciales, las apuntamos y las sumamos.

El psicólogo Daniel Kahneman (Pensar rápido, pensar despacio) popularizó esta distinción hablando de un pensamiento rápido e intuitivo y otro lento y deliberado. Para el segundo necesitamos tiempo y, casi siempre, un sitio donde apuntar.

Cuando pedimos a un modelo de lenguaje que conteste solo con el resultado, lo obligamos a trabajar siempre del primer modo: el primer token que escriba ya tiene que ser la respuesta. Para preguntas de memoria funciona muy bien. Para problemas de varios pasos, no tanto. Vamos a comprobarlo.

## **2. Preparar los experimentos**

Para ejecutar las celdas de este notebook necesitas:

1. **LM Studio abierto y con el servidor local activo** (sección **Model Server**), como vimos en el notebook 08.
2. **Dos modelos descargados**: `qwen/qwen3-4b-2507`, que responde directamente, y `qwen/qwen3-4b-thinking-2507`, la versión *Thinking* del mismo modelo, que usaremos en el apartado 7. No hace falta cargarlos a mano: LM Studio los carga en memoria cuando llega la primera petición.
3. **La biblioteca `openai` de Python**. Si no la tienes, instálala escribiendo `pip install openai` en una terminal.

Como vimos en el notebook anterior, LM Studio imita la API de OpenAI, así que usaremos la biblioteca `openai` y la **API Responses**. El mismo código serviría para hablar con los modelos de OpenAI en la nube: bastaría con cambiar la dirección y el nombre del modelo.

Igual que en el notebook anterior, estas celdas no funcionan en Google Colab, porque Colab no puede conectar con el LM Studio de tu ordenador. Descarga el notebook y ábrelo con Jupyter en el mismo equipo en el que tienes LM Studio.

La primera celda crea la conexión y comprueba que el servidor responde:

In [1]:
from openai import OpenAI

# LM Studio no exige una clave real, pero la biblioteca necesita que escribamos alguna.
cliente = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

MODELO = "qwen/qwen3-4b-2507"                      # responde directamente
MODELO_RAZONADOR = "qwen/qwen3-4b-thinking-2507"   # razona antes de responder

try:
    disponibles = [modelo.id for modelo in cliente.models.list()]
    print("Conexión correcta. Modelos disponibles en LM Studio:")
    for nombre in disponibles:
        print("  -", nombre)
except Exception as error:
    print("No se ha podido conectar con LM Studio.")
    print("Comprueba que está abierto y que el servidor local está en marcha.")
    print("Detalle:", error)

Conexión correcta. Modelos disponibles en LM Studio:
  - qwen/qwen3-4b-thinking-2507
  - google/gemma-4-e4b
  - qwen3.5-4b
  - text-embedding-nomic-embed-text-v1.5
  - qwen/qwen3-4b-2507


Si en tu lista los modelos tienen otro identificador, cámbialo en las variables `MODELO` y `MODELO_RAZONADOR`.

Para no repetir código, definimos una pequeña función que envía una pregunta y devuelve la respuesta de la API. Fíjate en la **temperatura 0**: como vimos en LM Studio, con temperatura baja el modelo elige casi siempre el token más probable. Así, las diferencias que observemos se deberán a cómo preguntamos y no al azar.

In [2]:
def preguntar(pregunta, modelo=MODELO, temperatura=0):
    """Envía una pregunta al modelo y devuelve la respuesta de la API."""
    return cliente.responses.create(model=modelo, input=pregunta, temperature=temperatura)

## **3. Dónde falla la respuesta directa**

Empecemos con un problema que cualquier alumno de primaria resolvería con papel y lápiz:

> Una profesora compra 23 cajas con 12 rotuladores cada una. Reparte 8 rotuladores a cada uno de sus 27 alumnos. ¿Cuántos rotuladores le sobran?

La solución requiere tres operaciones: 23 × 12 = 276 rotuladores comprados, 27 × 8 = 216 repartidos y 276 − 216 = **60** sobrantes.

Se lo planteamos al modelo pidiéndole solo el resultado:

In [3]:
problema = ("Una profesora compra 23 cajas con 12 rotuladores cada una. "
            "Reparte 8 rotuladores a cada uno de sus 27 alumnos. "
            "¿Cuántos rotuladores le sobran?")

DIRECTA = " Responde únicamente con el resultado, sin explicación."

respuesta_directa = preguntar(problema + DIRECTA)
print(respuesta_directa.output_text)

14


El modelo responde **14**. La respuesta correcta es 60.

No es que el modelo «no sepa multiplicar». Lo que ocurre es que le hemos pedido algo muy difícil: que el primer token que genere sea ya el resultado final de tres operaciones encadenadas. Para acertar tendría que calcular 276 y 216 «de cabeza», en un solo paso de cálculo de la red, sin poder apuntarlos en ningún sitio. Lo que produce es un número con aspecto razonable para un problema de este tipo. Es el mismo fenómeno que ya conocemos: una continuación verosímil que no es correcta.

## **4. Pensar paso a paso: la cadena de pensamiento**

Hagamos un único cambio. Mismo modelo, mismo problema, misma temperatura. Solo cambiamos la frase final de la petición:

In [4]:
PASO_A_PASO = " Razona paso a paso y termina con una línea que empiece por «Respuesta:»."

respuesta_pasos = preguntar(problema + PASO_A_PASO)
print(respuesta_pasos.output_text)

Vamos a resolver el problema paso a paso:

1. **Cálculo total de rotuladores comprados:**

   La profesora compra 23 cajas, y cada caja tiene 12 rotuladores.

   Entonces, el número total de rotuladores es:

   \[
   23 \times 12 = 276
   \]

2. **Cálculo total de rotuladores repartidos a los alumnos:**

   Tiene 27 alumnos, y reparte 8 rotuladores a cada uno.

   Entonces, el número total de rotuladores repartidos es:

   \[
   27 \times 8 = 216
   \]

3. **Cálculo de los rotuladores que le sobran:**

   Restamos los rotuladores repartidos del total comprado:

   \[
   276 - 216 = 60
   \]

Por lo tanto, le sobran 60 rotuladores.

Respuesta: 60


Ahora la respuesta es **60**, la correcta. (Los símbolos `\[`, `\times` y `\]` son la forma en la que el modelo escribe fórmulas en LaTeX; en una aplicación de chat se verían como expresiones matemáticas.)

El modelo no ha aprendido nada entre una celda y la otra. Sus parámetros son exactamente los mismos. Lo único que ha cambiado es que le hemos dejado escribir los pasos intermedios antes de dar el resultado.

Esta técnica se llama **cadena de pensamiento** o, en inglés, ***chain of thought*** (abreviado CoT). La describió en 2022 un grupo de investigadores de Google, que mostraron que los modelos resolvían mucho mejor los problemas de aritmética y de lógica si en la petición se incluían ejemplos resueltos paso a paso. Ese mismo año, otro trabajo comprobó que ni siquiera hacían falta los ejemplos: bastaba con añadir a la pregunta la frase *«Let's think step by step»* («Pensemos paso a paso»). En una de las colecciones de problemas aritméticos que utilizaron, esa sola frase hizo que el modelo pasara de acertar aproximadamente el 18 % de los problemas a acertar el 79 %.

<div style="display:inline-block; background-color:#fff3cd; border:1px solid #f0c36d; border-left:6px solid #e0a800; border-radius:6px; padding:12px 16px; margin:12px 0; color:#333;">
<b>La cadena de pensamiento consiste en pedir al modelo que escriba los pasos intermedios de su razonamiento antes de dar la respuesta final. El modelo es el mismo; lo que cambia es que ahora dispone de un sitio donde apuntar.</b>
</div>

### Un pequeño experimento

Un solo ejemplo podría ser casualidad. Probemos con cinco problemas, cada uno planteado de las dos maneras. La función `respuesta_final` se queda con lo que el modelo escribe después de «Respuesta:».

In [5]:
problemas = {
    "Rotuladores": (problema, "60"),
    "Excursión": ("En un instituto hay 4 grupos de 1.º de ESO con 28 alumnos cada uno y 3 grupos de 2.º de ESO "
                  "con 31 alumnos cada uno. A una excursión va el 75 % del alumnado de 1.º y dos tercios del "
                  "alumnado de 2.º. ¿Cuántos alumnos van a la excursión?", "146"),
    "Horario": ("Una excursión sale a las 9:40. El viaje tiene tres trayectos de 47, 68 y 55 minutos, con dos "
                "paradas de 25 minutos entre ellos. ¿A qué hora llega?", "13:20"),
    "Edades": ("Marta tiene el triple de edad que su hijo Leo. Dentro de 12 años tendrá el doble. "
               "¿Cuántos años tiene Marta ahora?", "36"),
    "Depósito": ("Un depósito de 500 litros empieza lleno y pierde cada día un 10 % de lo que contiene. "
                 "¿Cuántos litros quedan después de 3 días?", "364,5"),
}

def respuesta_final(texto):
    """Devuelve lo que el modelo escribe después de «Respuesta:»."""
    return texto.split("Respuesta:")[-1].replace("*", "").strip()

print(f"{'Problema':<12} {'Correcta':<9} {'Directa':<14} Paso a paso")
print("-" * 80)
for nombre, (enunciado, correcta) in problemas.items():
    directa = preguntar(enunciado + DIRECTA).output_text.strip()
    pasos = respuesta_final(preguntar(enunciado + PASO_A_PASO).output_text)
    print(f"{nombre:<12} {correcta:<9} {directa:<14} {pasos}")

Problema     Correcta  Directa        Paso a paso
--------------------------------------------------------------------------------
Rotuladores  60        14             60
Excursión    146       157            146 alumnos van a la excursión.
Horario      13:20     16:30          13:20
Edades       36        48             36 años.
Depósito     364,5     348.29 litros  364.5 litros quedan después de 3 días.


El resultado es rotundo: **cinco fallos de cinco** con la respuesta directa y **cinco aciertos de cinco** razonando paso a paso. (El modelo escribe los decimales con punto, a la inglesa: 364.5 es nuestro 364,5.)

Observa también las respuestas directas equivocadas: 157 alumnos, las 16:30, 48 años… Ninguna es disparatada. Todas tienen el aspecto que tendría una respuesta correcta, y por eso son peligrosas: si no conociéramos la solución, nada en ellas nos haría sospechar.

## **5. Por qué funciona**

<div style="text-align:center">
<img src="https://raw.githubusercontent.com/cayetanoguerra/redes-neuronales/main/notebooks/imgs/razonamiento/razonamiento-01-directa-vs-pasos.png" alt="Con la respuesta directa no hay dónde apuntar; razonando, cada paso escrito se convierte en contexto para el siguiente" width="75%" style="max-width:100%;height:auto;border:1px solid #dae1e8;border-radius:8px;">
</div>

<p style="text-align:center"><em>Con la respuesta directa no hay dónde apuntar; razonando, cada paso escrito se convierte en contexto para el siguiente</em></p>

La explicación está en la **generación autorregresiva** que estudiamos en el notebook 07: cada token que el modelo escribe se añade al contexto y se utiliza para calcular el siguiente.

Cuando el modelo escribe «23 × 12 = 276», ese 276 ya no tiene que recordarlo: está ahí, en el contexto, a la vista. Para escribir el paso siguiente solo tiene que resolver una operación sencilla, y para la resta final le basta con mirar los dos números que él mismo ha dejado escritos. Un problema difícil de tres pasos se ha convertido en tres problemas fáciles de un paso.

Dicho de otra forma:

- **El texto generado hace de memoria de trabajo.** Es el papel y el lápiz del modelo.
- **Más tokens significan más cálculo.** Cada token generado es una pasada completa por la red neuronal. Una respuesta de 2 tokens dispone de 2 pasadas para resolver el problema; una de 230 tokens dispone de 230.
- **Cada paso se parece a algo conocido.** Durante el entrenamiento, el modelo ha visto muchísimos textos donde aparece una multiplicación sencilla seguida de su resultado. Ha visto muchos menos donde un enunciado largo va seguido directamente de la solución.

<div style="display:inline-block; background-color:#fff3cd; border:1px solid #f0c36d; border-left:6px solid #e0a800; border-radius:6px; padding:12px 16px; margin:12px 0; color:#333;">
<b>Un modelo de lenguaje no puede pensar en silencio y luego contestar: razona escribiendo. Si no le dejamos escribir el razonamiento, no hay razonamiento.</b>
</div>

Esta idea tiene una consecuencia práctica inmediata. Cuando una tarea requiera varios pasos (corregir un ejercicio, comparar dos documentos, planificar una actividad), una instrucción como «responde solo sí o no» o «dame únicamente la nota» puede empeorar el resultado. Es preferible pedir primero el análisis y después la conclusión.

## **6. Variantes de la cadena de pensamiento**

A partir de la idea original surgieron pronto varias mejoras. Veamos tres.

### 6.1. Enseñar con un ejemplo (*few-shot*)

En lugar de dar la instrucción «razona paso a paso», podemos **mostrar un ejemplo resuelto** y dejar que el modelo lo imite. Así era, de hecho, la propuesta original de 2022. A esta forma de preguntar, en la que la petición incluye uno o varios ejemplos, se la llama ***few-shot*** («con unos pocos ejemplos»).

En la siguiente celda no hay ninguna instrucción sobre cómo razonar. Solo hay un problema resuelto y, a continuación, el problema de la excursión con la palabra «Razonamiento:» esperando continuación.

In [6]:
ejemplo = ("Pregunta: En una biblioteca hay 6 estanterías con 45 libros cada una. "
           "Se prestan 38 libros y se devuelven 15. ¿Cuántos libros hay ahora?\n"
           "Razonamiento: Al principio hay 6 × 45 = 270 libros. Tras los préstamos quedan 270 − 38 = 232. "
           "Con las devoluciones hay 232 + 15 = 247.\n"
           "Respuesta: 247\n\n")

nuevo_problema = problemas["Excursión"][0]

respuesta_ejemplo = preguntar(ejemplo + "Pregunta: " + nuevo_problema + "\nRazonamiento:")
print(respuesta_ejemplo.output_text)

Tenemos que calcular cuántos alumnos van a la excursión, considerando que:

- Hay 4 grupos de 1º de ESO con 28 alumnos cada uno.
- Hay 3 grupos de 2º de ESO con 31 alumnos cada uno.
- El 75% del alumnado de 1º va a la excursión.
- Dos tercios del alumnado de  2º va a la excursión.

**Paso 1: Calcular el número total de alumnos de 1º de ESO**

4 grupos × 28 alumnos = 112 alumnos en 1º de ESO.

El 75% de ellos va a la excursión:

75% de 112 = \( 0.75 \times 112 = 84 \) alumnos.

**Paso 2: Calcular el número total de alumnos de 2º de ESO**

3 grupos × 31 alumnos = 93 alumnos en 2º de ESO.

Dos tercios de ellos van a la excursión:

\( \frac{2}{3} \times 93 = 62 \) alumnos.

**Paso 3: Sumar los que van de 1º y de 2º**

84 + 62 = 146 alumnos.

**Respuesta final:**

146 alumnos van a la excursión.  

✅ Respuesta: **146**


El modelo ha entendido lo que se esperaba de él: ha razonado por pasos y ha llegado a 146, la respuesta correcta, sin que se lo pidiéramos con palabras. No ha copiado el formato compacto de nuestro ejemplo (ha preferido su estilo habitual, con títulos y negritas), pero sí la idea de razonar antes de responder.

Los ejemplos son especialmente útiles cuando queremos que el razonamiento siga un método concreto, por ejemplo los criterios de una rúbrica de corrección.

### 6.2. Preguntar varias veces y votar (autoconsistencia)

Con temperatura 0 el modelo da casi siempre la misma respuesta. Si subimos la temperatura, cada ejecución puede seguir un camino distinto. ¿Y si aprovechamos esa variedad? La idea de la **autoconsistencia** (*self-consistency*) es sencilla: hacemos la misma pregunta varias veces, dejamos que el modelo razone de formas diferentes y nos quedamos con **la respuesta final más repetida**. Si varios caminos distintos llegan al mismo resultado, es más probable que sea el correcto.

Probémoslo con un acertijo clásico, cuya respuesta correcta es 8 días (al final del día 7 el caracol está a 7 metros; el día 8 sube 3 y sale):

In [7]:
from collections import Counter

BREVE = (" Razona brevemente, en tres o cuatro frases como máximo, "
         "y termina con una línea que empiece por «Respuesta:».")

def votar(pregunta, intentos=5):
    """Hace la misma pregunta varias veces con temperatura 0,8 y cuenta las respuestas finales."""
    cadenas = []
    finales = []
    for i in range(intentos):
        texto = preguntar(pregunta + BREVE, temperatura=0.8).output_text
        final = respuesta_final(texto).rstrip(".")
        cadenas.append(texto)
        finales.append(final)
        print(f"Intento {i + 1}: {final}")
    ganadora, votos = Counter(finales).most_common(1)[0]
    print(f"\nRespuesta más votada: {ganadora} ({votos} de {intentos})")
    return cadenas

caracol = ("Un caracol está en el fondo de un pozo de 10 metros. Cada día sube 3 metros y cada noche "
           "resbala 2. ¿Cuántos días tarda en salir del pozo?")

cadenas_caracol = votar(caracol)

Intento 1: 8 días
Intento 2: 7 días
Intento 3: 8 días
Intento 4: 8 días
Intento 5: 8 días

Respuesta más votada: 8 días (4 de 5)


En esta ejecución, cuatro de los cinco intentos responden 8 días y uno responde 7. Si solo hubiéramos preguntado una vez, podría habernos tocado el intento equivocado. Con la votación, el error queda en minoría. Como aquí interviene el azar, al ejecutar la celda en tu ordenador obtendrás otras cadenas y quizá otro reparto de votos.

Leamos el intento que falló:

In [8]:
print(cadenas_caracol[1])

Cada día neto sube 1 metro (3 metros de día menos 2 de noche), pero al salir del pozo no vuelve a resbalar. Al día 7 sube hasta los 10 metros, pero como solo necesita salir, no resbala esa noche.  
Así, al final del séptimo día, ya ha salido del pozo.  
Respuesta: 7 días


El razonamiento empieza bien (un metro neto al día, y el último día ya no resbala), pero después afirma que el día 7 el caracol llega a los 10 metros, lo que no es cierto. Un solo paso incorrecto y toda la cadena termina en una respuesta equivocada, expresada con la misma seguridad que las demás.

La autoconsistencia tiene un precio evidente: cinco intentos cuestan cinco veces más tiempo y más tokens. Y, como veremos en el apartado 8, no lo arregla todo.

### 6.3. Explorar varios caminos (*Tree of Thoughts*)

Una cadena de pensamiento es una línea recta: si un paso intermedio es malo, todo lo que viene después hereda el error, como acabamos de ver. En 2023 se propuso el **árbol de pensamientos** (*Tree of Thoughts*): en lugar de una sola cadena, el sistema genera varias continuaciones posibles en cada paso, las evalúa, sigue adelante por las más prometedoras y **vuelve atrás** cuando un camino no lleva a ningún sitio. Se parece a cómo exploramos las jugadas posibles en una partida de ajedrez.

No lo programaremos aquí, pero quédate con la idea: ya no es una sola petición al modelo, sino un **programa que organiza muchas peticiones**. Esa forma de trabajar, con un programa que dirige al modelo paso a paso, es el germen de los **agentes** que estudiaremos más adelante.

## **7. Modelos razonadores**

Hasta aquí, el razonamiento era cosa nuestra: teníamos que acordarnos de escribir «razona paso a paso». La siguiente pregunta era natural: si razonar antes de responder funciona tan bien, ¿por qué no **entrenar a los modelos para que lo hagan siempre por sí mismos**?

Eso son los **modelos razonadores**. En septiembre de 2024 OpenAI presentó o1, el primer modelo de gran difusión entrenado para generar un razonamiento antes de cada respuesta. En enero de 2025, DeepSeek-R1 hizo lo mismo con pesos abiertos y mostrando el razonamiento completo al usuario. Hoy casi todas las familias de modelos tienen versiones razonadoras, incluidas las que podemos ejecutar en local: es lo que indican la etiqueta **Thinking** en el nombre de un modelo y el icono 💡 **Reasoning** de LM Studio que vimos en el notebook 08.

<div style="text-align:center">
<img src="https://raw.githubusercontent.com/cayetanoguerra/redes-neuronales/main/notebooks/imgs/razonamiento/razonamiento-02-modelo-razonador.png" alt="Un modelo razonador genera primero un razonamiento y, a continuación, la respuesta que ve el usuario" width="75%" style="max-width:100%;height:auto;border:1px solid #dae1e8;border-radius:8px;">
</div>

<p style="text-align:center"><em>Un modelo razonador genera primero un razonamiento y, a continuación, la respuesta que ve el usuario</em></p>

El funcionamiento no encierra ningún misterio nuevo. El modelo sigue generando token a token, pero ha sido entrenado para producir primero un bloque de razonamiento, separado por unas marcas especiales, y después la respuesta. Los tokens de ese bloque se llaman **tokens de razonamiento**.

Vamos a verlo. Enviamos el problema de los rotuladores a la versión *Thinking* de Qwen3, **tal cual, sin ninguna instrucción sobre cómo responder**. La respuesta de la API llega ahora en dos partes: un elemento de tipo `reasoning` con el razonamiento y un elemento de tipo `message` con la respuesta.

In [9]:
respuesta_razonador = preguntar(problema, modelo=MODELO_RAZONADOR)

for elemento in respuesta_razonador.output:
    if elemento.type == "reasoning":
        print("=== RAZONAMIENTO ===")
        print("".join(parte.text for parte in elemento.content).strip())
    elif elemento.type == "message":
        print()
        print("=== RESPUESTA ===")
        print(respuesta_razonador.output_text.strip())

=== RAZONAMIENTO ===
Okay, let's see. So the problem is: A teacher buys 23 boxes with 12 pens each. She distributes 8 pens to each of her 27 students. How many pens does she have left over?

Hmm, first I need to figure out the total number of pens she bought. Since each box has 12 pens and she bought 23 boxes, I should multiply 23 by 12. Let me do that.

23 times 12. Let's calculate that. 20 times 23 is 460, and 3 times 23 is 69, so 460 + 69 = 529. Wait, no, wait. Wait, 23 times 12. Maybe better to do 23*10=230 and 23*2=46, so 230+46=276. Oh, right! Because 12 is 10+2. So 23*12=276. Yeah, that's correct. Let me check: 20*12=240, 3*12=36, so 240+36=276. Yes, that's right. So total pens are 276.

Then she gives 8 pens to each of her 27 students. So total pens distributed would be 27 students * 8 pens each. Let's calculate that. 27*8. 20*8=160, 7*8=56, so 160+56=216. So she gives out 216 pens.

Now, the number of pens left over would be total pens minus distributed pens: 276 - 216. Let's 

Merece la pena leer el razonamiento con calma, porque enseña mucho:

- **Nadie le ha pedido que razone.** Lo hace porque ha sido entrenado para ello.
- **Razona en inglés**, aunque la pregunta y la respuesta estén en español. Es frecuente en estos modelos: el bloque de razonamiento es un borrador para uso propio y no está pensado para el lector.
- **Se equivoca y se corrige.** Al calcular 23 × 12 escribe primero «460 + 69 = 529» y, acto seguido, «*Wait, no, wait*»: detecta el error, repite la operación de otra manera y obtiene 276.
- **Comprueba el resultado** dos veces antes de darlo por bueno.

Ese ir y venir (probar, dudar, rehacer, verificar) es justo lo que no cabe en una respuesta directa, y es lo que hace más fiables a estos modelos en problemas difíciles.

### Cuánto razonar

Razonar cuesta tiempo. Por eso muchos modelos razonadores permiten regular el **esfuerzo de razonamiento**. En la API de OpenAI se hace con el parámetro `reasoning`:

```python
cliente_openai = OpenAI()   # usa la clave de OpenAI guardada en la variable OPENAI_API_KEY

respuesta = cliente_openai.responses.create(
    model="NOMBRE_DEL_MODELO",        # un modelo razonador de OpenAI
    input=problema,
    reasoning={"effort": "low"},      # por ejemplo "low", "medium" o "high"
)
```

Con un esfuerzo bajo el modelo razona poco y contesta rápido; con uno alto dedica muchos más tokens a pensar. LM Studio acepta este mismo parámetro con los modelos que admiten varios niveles. Nuestro Qwen3 *Thinking* no es uno de ellos: razona siempre.

Hay otra diferencia que conviene conocer. Nuestro modelo local nos entrega su razonamiento completo. Los servicios comerciales, en cambio, suelen ocultarlo y muestran, como mucho, un resumen. Los tokens de razonamiento, eso sí, se generan y se cobran igualmente.

<div style="display:inline-block; background-color:#fff3cd; border:1px solid #f0c36d; border-left:6px solid #e0a800; border-radius:6px; padding:12px 16px; margin:12px 0; color:#333;">
<b>Un modelo razonador es un modelo de lenguaje entrenado para generar un razonamiento antes de cada respuesta. Lo que antes conseguíamos escribiendo «razona paso a paso» ahora forma parte del propio modelo.</b>
</div>

## **8. Costes y límites**

### Razonar no es gratis

La API nos dice cuántos tokens ha generado el modelo en cada respuesta. Comparemos las tres formas en las que hemos resuelto el problema de los rotuladores:

In [10]:
print(f"{'Forma de responder':<28} {'Tokens generados':>16} {'De razonamiento':>16}")
print("-" * 62)
for nombre, r in [("Directa", respuesta_directa),
                  ("Paso a paso", respuesta_pasos),
                  ("Modelo razonador", respuesta_razonador)]:
    uso = r.usage
    print(f"{nombre:<28} {uso.output_tokens:>16} {uso.output_tokens_details.reasoning_tokens:>16}")

Forma de responder           Tokens generados  De razonamiento
--------------------------------------------------------------
Directa                                     2                0
Paso a paso                               230                0
Modelo razonador                          822              632


La respuesta directa ocupó 2 tokens. La cadena de pensamiento, 230. El modelo razonador generó 822, de los cuales 632 fueron de razonamiento. La última respuesta necesitó más de 400 veces más tokens que la primera.

Como el modelo genera los tokens uno a uno, más tokens significan **más tiempo de espera**. En el ordenador en el que se preparó este notebook, la respuesta directa tardó alrededor de medio segundo y la cadena de pensamiento, unos cinco. En un servicio de pago significan, además, **más coste**, porque se factura por token.

Por eso razonar no siempre compensa:

| Tipo de tarea | ¿Merece la pena razonar? |
|---|---|
| Datos sencillos, traducciones, reformular un texto | No: la respuesta directa es igual de buena y mucho más rápida |
| Problemas de varios pasos, cálculos, lógica | Sí |
| Analizar un documento, corregir con criterios, planificar | Sí |
| Conversación informal | No |

### Razonar no garantiza acertar

Probemos la votación del apartado 6.2 con una tarea de otro tipo: contar letras. La respuesta correcta es **10** (perro, 2; Ramiro, 2; corre, 2; por, 1; carretera, 3).

In [11]:
letras = "¿Cuántas veces aparece la letra «r» en la frase «El perro de Ramiro corre por la carretera»?"

cadenas_letras = votar(letras)

Intento 1: 10 veces
Intento 2: 9 veces
Intento 3: 11 veces
Intento 4: 11 veces
Intento 5: 9 veces

Respuesta más votada: 9 veces (2 de 5)


Esta vez la votación no nos salva: las respuestas se reparten entre 9, 10 y 11, ninguna alcanza la mayoría y la que el programa da por ganadora es incorrecta. La autoconsistencia corrige los despistes ocasionales, pero no un **error sistemático**. Y contar letras es una debilidad conocida de estos modelos: recuerda que el modelo no ve letras, sino tokens, y que una palabra como «carretera» puede llegarle troceada en dos o tres fragmentos.

Leamos ahora el único intento que acertó:

In [12]:
print(cadenas_letras[0])

Analizamos la frase palabra por palabra: «El perro de Ramiro corre por la carretera».  
Contamos las «r»: en «perro» hay 1 (p-e-r-r-o), en «Ramiro» hay 3 (R-a-m-i-r-o), en «corre» hay 2 (c-o-r-r-e), en «por» hay 1 (p-o-r), en «carretera» hay 3 (c-a-r-r-e-t-e-r-a).  
Sumando: 1 + 3 + 2 + 1 + 3 = 10 letras «r».  
Respuesta: 10 veces


La respuesta final es correcta, pero el razonamiento no lo es: dice que en «perro» hay una sola «r» (hay dos) y que en «Ramiro» hay tres (hay dos). Los dos errores se compensan y la suma da 10 por casualidad.

Este ejemplo ilustra dos límites importantes:

- **Los errores se arrastran.** Cada paso se apoya en los anteriores. Un dato mal calculado al principio contamina todo lo que viene después, y el modelo rara vez vuelve atrás a revisarlo.
- **El razonamiento escrito no es una prueba.** Que una explicación parezca ordenada y termine en el resultado correcto no garantiza que cada paso sea verdadero. Varios estudios han mostrado, además, que el razonamiento que un modelo escribe no siempre refleja fielmente lo que de verdad ha determinado su respuesta. Conviene leerlo como una explicación plausible, no como una demostración.

Queda una última limitación, quizá la más importante. Razonando, el modelo sigue disponiendo únicamente de lo que aprendió durante el entrenamiento y de lo que hay en el contexto. Puede pensar mucho, pero **no puede consultar un dato actual, ni usar una calculadora, ni contar las letras con un programa**. Para eso necesita herramientas, y a ellas dedicaremos el próximo notebook.

## **Resumen**

- Un modelo de lenguaje solo conserva, de un paso al siguiente, el texto que ya ha escrito. Si le pedimos únicamente el resultado, tiene que acertarlo sin cálculos intermedios.
- La **cadena de pensamiento** (*chain of thought*) consiste en pedirle que escriba los pasos antes de la respuesta. En nuestro experimento, el mismo modelo pasó de fallar cinco problemas de cinco a acertarlos todos.
- Funciona porque **el texto generado hace de memoria de trabajo**: cada paso escrito pasa a formar parte del contexto del siguiente.
- Se puede inducir con una instrucción, con ejemplos resueltos (***few-shot***), repitiendo la pregunta y votando (**autoconsistencia**) o explorando varios caminos (***Tree of Thoughts***).
- Los **modelos razonadores** han sido entrenados para generar un razonamiento antes de cada respuesta, sin que haya que pedírselo, y muchos permiten regular el esfuerzo.
- Razonar cuesta **tokens y tiempo**, no compensa en tareas sencillas y **no garantiza acertar**: los errores se arrastran y el razonamiento escrito no es una demostración.

## **Ejercicios**

1. **Un problema de tu asignatura.** Escribe un problema de varios pasos de tu materia y plantéaselo al modelo de las dos formas, con `DIRECTA` y con `PASO_A_PASO`. ¿Falla la respuesta directa? ¿En qué paso se equivocaría un alumno?
2. **Votar sin azar.** En la función `votar`, cambia `temperatura=0.8` por `temperatura=0` y ejecútala con el acertijo del caracol. ¿Qué ocurre con los cinco intentos? ¿Por qué deja de tener sentido la votación?
3. **Más votos.** Vuelve a la temperatura 0,8 y ejecuta `votar(caracol, intentos=9)`. ¿Cambia la respuesta ganadora? ¿Cuánto más tarda?
4. **El razonador ante las letras.** Envía la pregunta de las letras al modelo razonador con `preguntar(letras, modelo=MODELO_RAZONADOR)` y lee su razonamiento. ¿Acierta? ¿Cuántos tokens de razonamiento ha necesitado?
5. **Razonar de más.** Pregunta al modelo razonador «¿Cuál es la capital de Francia?» y mira cuántos tokens de razonamiento ha generado. ¿Qué conclusión sacas sobre cuándo conviene usar un modelo razonador?

## **Glosario breve**

**Cadena de pensamiento (*chain of thought*, CoT).** Técnica que consiste en pedir al modelo que escriba los pasos intermedios de su razonamiento antes de dar la respuesta final.

***Few-shot*.** Forma de preguntar en la que la petición incluye uno o varios ejemplos resueltos para que el modelo los imite.

**Autoconsistencia (*self-consistency*).** Técnica que repite la misma pregunta varias veces con algo de azar y se queda con la respuesta final más frecuente.

**Árbol de pensamientos (*Tree of Thoughts*).** Método en el que un programa hace que el modelo genere y evalúe varios caminos de razonamiento, y retrocede cuando uno no funciona.

**Modelo razonador.** Modelo de lenguaje entrenado para generar un razonamiento antes de cada respuesta.

**Tokens de razonamiento.** Tokens que un modelo razonador genera mientras razona, antes de la respuesta visible. Consumen tiempo y, en los servicios de pago, se facturan.

**Esfuerzo de razonamiento.** Ajuste que regula cuánto razona un modelo antes de responder.

**API Responses.** Interfaz actual de OpenAI para pedir respuestas a un modelo desde un programa. LM Studio ofrece una interfaz compatible para los modelos locales.

[← Notebook 9](https://colab.research.google.com/github/cayetanoguerra/redes-neuronales/blob/main/notebooks/09-api.ipynb)